# Renal ICU Transfer Prediction — Fully Reproducible Research Notebook

**Dataset:** MIMIC-IV v3.1  
**Clinical task:** Among adult admissions retrospectively coded for AKI/CKD and still outside the ICU at 24 hours, predict the *observed first ICU transfer* occurring after 24 hours and by 72 hours.

This notebook exposes the **actual code from the completed seven-stage pipeline**, rather than wrapping it in a compressed launcher. It is intended as a transparent reference architecture for comparisons with other machine-learning models. It includes the exploratory Phase 3 training track and the cleaner Phase 4B track. Read the methodological notes before interpreting results.

**Research use only.** Observed ICU transfer is not the same as clinical ICU need, benefit from ICU care, or a valid triage recommendation.

## How to run

1. Install packages in the Jupyter kernel environment (next cell).
2. Edit `DATA_ROOT` and `OUTPUT_ROOT` if needed.
3. Run cells in order. The laboratory extraction scans ~158 million records and can take significant time. Enable `REUSE_EXISTING = True` to skip any phase whose expected output files already exist.
4. Outputs are written to separate folders, so competing architectures can use the same cohort and feature files.

**Important for fair comparisons:** Use exactly the same patient-level splits, inclusion criteria, feature availability rules, and evaluation metrics for each architecture. Save patient IDs for each split. Never tune on the final evaluation set.

In [ ]:
import os
# Run once if packages are missing (uncomment):
# %pip install numpy pandas pyarrow scikit-learn lightgbm catboost shap matplotlib joblib
from pathlib import Path
from types import SimpleNamespace
import json
DATA_ROOT = Path(os.environ["MIMIC_PROJECT_ROOT"])
OUTPUT_ROOT = DATA_ROOT / 'end_to_end_results'
REUSE_EXISTING = True
THREADS = 4
SHAP_SAMPLE = 500
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('MIMIC-IV root:', DATA_ROOT)
print('Output root:', OUTPUT_ROOT)

## Phase 1 — Cohort construction and label definition

**Question:** Which hospital admissions belong to the retrospective renal cohort, and which have an observable ICU-transfer outcome?

The script reads patients, admissions, ICD diagnoses and ICU stays. It calculates age using `anchor_age`, `anchor_year` and admission year; identifies AKI (ICD-9 584.*, ICD-10 N17.*) and CKD (ICD-9 585.*, ICD-10 N18.*); and establishes the prediction landmark at admission + 24 hours.

Admissions already in ICU during the observation window or not alive/in hospital at the landmark are excluded. Positive labels correspond to first ICU entry after 24 hours and at or before 72 hours. Patients without definitive 72-hour observation are excluded from the binary analysis. The output separates **predictors**, **labels** and a restricted **audit** table containing future timestamps.

**Key caveats:** Discharge ICD codes may be assigned after the landmark; this is a retrospective diagnosis-defined cohort. Early discharge/death exclusion can introduce selection bias, and ICU transfer is a utilization event rather than proof of medical need.

**Expected prior run:** 59,949 labeled admissions; 3,015 positive; 30,904 unique patients.

**Implementation:** The next code cell contains the complete original phase implementation (imports, constants, helper functions and model logic). It is intentionally visible and editable so you can inspect every operation.

In [ ]:
#!/usr/bin/env python3
"""Phase 1: retrospective renal ICU-transfer landmark cohort for MIMIC-IV 3.1."""
import argparse
import json
from pathlib import Path
import pandas as pd
import numpy as np


def locate(base, module, stem):
    roots = [base, base / 'physionet.org' / 'files' / 'mimiciv' / '3.1', base / 'mimiciv' / '3.1']
    for root in roots:
        for suffix in ('.csv.gz', '.csv'):
            p = root / module / (stem + suffix)
            if p.is_file():
                return p
    raise FileNotFoundError(f'Cannot locate {module}/{stem}.csv[.gz] under {base}')


def load(base, module, stem, columns, dates=()):
    p = locate(base, module, stem)
    print('Reading', p, flush=True)
    df = pd.read_csv(p, usecols=columns, low_memory=False)
    for c in dates:
        df[c] = pd.to_datetime(df[c], errors='coerce')
    return df


def assert_unique(df, col, label):
    if df[col].isna().any() or df[col].duplicated().any():
        raise ValueError(f'{label}: {col} must be unique and non-null')


def build(base, output):
    output.mkdir(parents=True, exist_ok=True)
    patients = load(base, 'hosp', 'patients', ['subject_id','gender','anchor_age','anchor_year'])
    adm = load(base, 'hosp', 'admissions', ['subject_id','hadm_id','admittime','dischtime','deathtime','admission_type','admission_location'], ['admittime','dischtime','deathtime'])
    dx = load(base, 'hosp', 'diagnoses_icd', ['subject_id','hadm_id','icd_code','icd_version'])
    icu = load(base, 'icu', 'icustays', ['subject_id','hadm_id','stay_id','intime','outtime'], ['intime','outtime'])
    assert_unique(patients, 'subject_id', 'patients')
    assert_unique(adm, 'hadm_id', 'admissions')
    assert_unique(icu, 'stay_id', 'icustays')
    report = {}
    def count(name, df):
        report[name] = int(len(df))
        print(f'{name}: {len(df):,}', flush=True)

    count('raw_admissions', adm)
    adm = adm.loc[adm.admittime.notna() & adm.dischtime.notna() & (adm.dischtime > adm.admittime)].copy()
    count('valid_hospital_intervals', adm)
    # Fail closed on conflicting subject links.
    adm = adm.merge(patients, on='subject_id', how='inner', validate='many_to_one')
    adm['age'] = adm.anchor_age + adm.admittime.dt.year - adm.anchor_year
    adm = adm.loc[adm.age.ge(18) & adm.age.notna()].copy()
    count('adult_admissions', adm)

    dx['icd_code'] = dx.icd_code.astype('string').str.upper().str.replace('.', '', regex=False).str.strip()
    version = pd.to_numeric(dx.icd_version, errors='coerce')
    aki = ((version.eq(9) & dx.icd_code.str.startswith('584', na=False)) |
           (version.eq(10) & dx.icd_code.str.startswith('N17', na=False)))
    ckd = ((version.eq(9) & dx.icd_code.str.startswith('585', na=False)) |
           (version.eq(10) & dx.icd_code.str.startswith('N18', na=False)))
    renal = dx.loc[aki | ckd, ['subject_id','hadm_id']].copy()
    renal['AKI'] = aki.loc[renal.index].astype('int8')
    renal['CKD'] = ckd.loc[renal.index].astype('int8')
    renal = renal.groupby(['subject_id','hadm_id'], as_index=False)[['AKI','CKD']].max()
    adm = adm.merge(renal, on=['subject_id','hadm_id'], how='inner', validate='one_to_one')
    count('adult_retrospective_renal_admissions', adm)
    adm['kidney_group'] = np.select([adm.AKI.eq(1)&adm.CKD.eq(1), adm.AKI.eq(1), adm.CKD.eq(1)], ['AKI + CKD','AKI only','CKD only'], default='unknown')

    # ICU data integrity: exclude admissions with ANY malformed/conflicting ICU record,
    # rather than incorrectly treating them as non-ICU admissions.
    valid_icu = (icu.hadm_id.notna() & icu.subject_id.notna() & icu.intime.notna() & icu.outtime.notna() & (icu.outtime > icu.intime))
    malformed_keys = set(icu.loc[~valid_icu & icu.hadm_id.notna(), 'hadm_id'])
    icu = icu.loc[valid_icu].copy()
    joined = icu.merge(adm[['hadm_id','subject_id','admittime','dischtime']], on='hadm_id', how='inner', suffixes=('_icu','_adm'), validate='many_to_one')
    inconsistent = joined.subject_id_icu.ne(joined.subject_id_adm) | joined.intime.lt(joined.admittime) | joined.intime.gt(joined.dischtime) | joined.outtime.lt(joined.admittime)
    malformed_keys.update(joined.loc[inconsistent, 'hadm_id'].tolist())
    adm = adm.loc[~adm.hadm_id.isin(malformed_keys)].copy()
    count('after_excluding_malformed_icu_links', adm)
    icu = icu.loc[~icu.hadm_id.isin(malformed_keys)].copy()
    icu = icu.loc[icu.hadm_id.isin(adm.hadm_id)].copy()
    first = icu.sort_values(['hadm_id','intime','stay_id']).drop_duplicates('hadm_id')[['hadm_id','intime']].rename(columns={'intime':'first_icu_intime'})
    adm = adm.merge(first, on='hadm_id', how='left', validate='one_to_one')
    adm['landmark_time'] = adm.admittime + pd.Timedelta(hours=24)
    adm['horizon_end'] = adm.admittime + pd.Timedelta(hours=72)
    # Death timestamp is an additional conservative eligibility check; discharge
    # remains the primary observation boundary.
    adm = adm.loc[adm.dischtime.gt(adm.landmark_time) & (adm.deathtime.isna() | adm.deathtime.gt(adm.landmark_time))].copy()
    count('hospitalized_and_alive_at_24h', adm)
    adm = adm.loc[adm.first_icu_intime.isna() | adm.first_icu_intime.gt(adm.landmark_time)].copy()
    count('not_in_icu_during_first_24h', adm)

    positive = adm.first_icu_intime.notna() & adm.first_icu_intime.le(adm.horizon_end)
    # Early discharge or death before 72h with no ICU is a competing event:
    # do not silently label it as a fully observed negative.
    fully_observed_negative = (adm.first_icu_intime.isna() | adm.first_icu_intime.gt(adm.horizon_end)) & adm.dischtime.ge(adm.horizon_end) & (adm.deathtime.isna() | adm.deathtime.ge(adm.horizon_end))
    adm['label_status'] = np.select([positive, fully_observed_negative], ['positive','negative'], default='censored')
    report['censored_competing_events'] = int(adm.label_status.eq('censored').sum())
    cohort = adm.loc[adm.label_status.ne('censored')].copy()
    cohort['icu_transfer_24_72h'] = cohort.label_status.eq('positive').astype('int8')
    count('final_labeled_admissions', cohort)
    report['positive_admissions'] = int(cohort.icu_transfer_24_72h.sum())
    report['negative_admissions'] = int((cohort.icu_transfer_24_72h == 0).sum())
    report['unique_patients'] = int(cohort.subject_id.nunique())
    report['prevalence'] = float(cohort.icu_transfer_24_72h.mean()) if len(cohort) else None

    # No future timestamps or outcome fields in the predictor-facing table.
    predictor_columns = ['subject_id','hadm_id','landmark_time','age','gender','admission_type','admission_location']
    predictor_table = cohort[predictor_columns].copy()
    labels = cohort[['subject_id','hadm_id','icu_transfer_24_72h']].copy()
    assert_unique(predictor_table, 'hadm_id', 'predictor_table')
    assert_unique(labels, 'hadm_id', 'labels')
    assert not (cohort.first_icu_intime.notna() & cohort.first_icu_intime.le(cohort.landmark_time)).any()
    assert (cohort.dischtime > cohort.landmark_time).all()
    assert not (cohort.icu_transfer_24_72h.eq(0) & cohort.dischtime.lt(cohort.horizon_end)).any()
    assert not set(['first_icu_intime','dischtime','deathtime','horizon_end','label_status','icu_transfer_24_72h']).intersection(predictor_table.columns)
    predictor_table.to_parquet(output/'phase1_landmark_predictors.parquet', index=False)
    labels.to_parquet(output/'phase1_labels.parquet', index=False)
    # Restricted to research workspace: audit contains post-landmark timestamps.
    cohort.to_parquet(output/'phase1_cohort_audit.parquet', index=False)
    (output/'phase1_report.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
    print('\nWrote outputs to:', output)
    print(json.dumps(report, indent=2))


In [ ]:
# Run phase1 (or reuse completed results).
if REUSE_EXISTING and ((OUTPUT_ROOT / 'phase1_outputs' / 'phase1_landmark_predictors.parquet').exists() and (OUTPUT_ROOT / 'phase1_outputs' / 'phase1_labels.parquet').exists() and (OUTPUT_ROOT / 'phase1_outputs' / 'phase1_cohort_audit.parquet').exists()):
    print('Skipping phase1: expected outputs already exist')
else:
    build(DATA_ROOT, OUTPUT_ROOT / 'phase1_outputs')

**Check the output**

In [ ]:
p = OUTPUT_ROOT / 'phase1_outputs'
print('Output files:')
for f in sorted(p.iterdir()):
    print(' -', f.name)
for report in sorted(p.glob('*.json')):
    print('\n', report.name)
    print(report.read_text()[:4000])

## Phase 2 — Longitudinal laboratory feature extraction

**Question:** What clinical laboratory information was genuinely available before the 24-hour landmark?

This phase scans the large `labevents` table in chunks, retaining selected analytes and admissions. Both collection/chart time and result storage time must be available by the landmark. Values undergo duplicate, unit and plausibility checks. Measurements are summarized in **0–6, 6–12 and 12–24 hour** windows, plus longitudinal statistics such as changes, slopes and observation counts. Missingness is retained as potentially informative rather than blindly filled.

**Why:** A single latest creatinine value loses trajectory information. Temporal windows may reveal rapid renal or systemic deterioration. Measurement frequency can itself reflect clinician behavior, so interpret it carefully.

**Expected prior run:** ~158.4 million rows scanned, 616 engineered columns, 56,191 admissions with at least one selected lab. The feature table should retain all 59,949 eligible admissions.

**Implementation:** The next code cell contains the complete original phase implementation (imports, constants, helper functions and model logic). It is intentionally visible and editable so you can inspect every operation.

In [ ]:
#!/usr/bin/env python3
"""Phase 2: memory-bounded, leakage-aware temporal lab features for MIMIC-IV 3.1.

Research only. Uses lab charttime AND storetime <= 24h landmark. When
storetime is absent, fails closed by default. Does not read outcome labels.
"""
import argparse
import json
from pathlib import Path
import numpy as np
import pandas as pd

LAB_MAP = {
 50912:'creatinine',51006:'bun',50983:'sodium',50971:'potassium',50902:'chloride',
 50882:'bicarbonate',50893:'calcium',50960:'magnesium',50970:'phosphate',
 50931:'glucose',50813:'lactate',50868:'anion_gap',51301:'wbc',51222:'hemoglobin',
 51221:'hematocrit',51265:'platelets',51277:'rdw',50885:'bilirubin_total',
 50862:'albumin',50861:'alt',50878:'ast',50863:'alkaline_phosphatase',
 51237:'inr',51274:'pt',51275:'ptt',50820:'ph',50818:'pco2',50821:'po2'
}
# Wide safety bounds, applied only after item/unit checks. No quantile clipping.
RANGES = {'albumin':(.5,10),'alkaline_phosphatase':(1,10000),'alt':(1,30000),
'anion_gap':(0,60),'ast':(1,40000),'bicarbonate':(3,60),
'bilirubin_total':(.1,100),'bun':(1,300),'calcium':(2,20),
'chloride':(50,150),'creatinine':(.1,30),'glucose':(10,2000),
'hematocrit':(5,75),'hemoglobin':(2,25),'inr':(.5,30),'lactate':(.1,40),
'magnesium':(.2,10),'pco2':(5,200),'ph':(6.5,8),'phosphate':(.2,20),
'platelets':(1,2000),'po2':(5,700),'potassium':(1.5,10),'pt':(5,150),
'ptt':(10,200),'rdw':(5,50),'sodium':(80,180),'wbc':(.1,500)}

def locate(base):
    for root in (base,base/'physionet.org'/'files'/'mimiciv'/'3.1',base/'mimiciv'/'3.1'):
        for suffix in ('.csv.gz','.csv'):
            path=root/'hosp'/('labevents'+suffix)
            if path.exists(): return path
    raise FileNotFoundError('Could not find hosp/labevents.csv[.gz] under '+str(base))

def run(base, phase1, output, chunk_size):
    output.mkdir(parents=True,exist_ok=True)
    pred=pd.read_parquet(phase1/'phase1_landmark_predictors.parquet')
    required={'subject_id','hadm_id','landmark_time'}
    if not required.issubset(pred.columns): raise ValueError('Missing Phase 1 columns: '+str(required-set(pred.columns)))
    if pred.hadm_id.isna().any() or pred.hadm_id.duplicated().any(): raise ValueError('Non-unique hadm_id')
    pred['landmark_time']=pd.to_datetime(pred.landmark_time,errors='raise')
    # Use an explicit join on BOTH identifiers, rejecting conflicting links.
    links=pred[['subject_id','hadm_id','landmark_time']].copy()
    source=locate(base)
    print('Reading in chunks:',source,flush=True)
    selected=[]
    counters={'rows_read':0,'selected_item_and_admission':0,'available_by_landmark':0,'valid_numeric_unit_and_range':0}
    units={}
    cols=['subject_id','hadm_id','itemid','charttime','storetime','valuenum','valueuom']
    for i,chunk in enumerate(pd.read_csv(source,usecols=cols,chunksize=chunk_size,low_memory=False),1):
        counters['rows_read']+=len(chunk)
        chunk=chunk.loc[chunk.itemid.isin(LAB_MAP)&chunk.hadm_id.isin(links.hadm_id)].copy()
        if chunk.empty: continue
        chunk=chunk.merge(links,on=['subject_id','hadm_id'],how='inner',validate='many_to_one')
        counters['selected_item_and_admission']+=len(chunk)
        if chunk.empty: continue
        chunk['charttime']=pd.to_datetime(chunk.charttime,errors='coerce')
        chunk['storetime']=pd.to_datetime(chunk.storetime,errors='coerce')
        # Strict timestamp policy: results must be charted AND stored by prediction time.
        chunk=chunk.loc[chunk.charttime.notna()&chunk.storetime.notna()&
            chunk.charttime.le(chunk.landmark_time)&chunk.storetime.le(chunk.landmark_time)&
            chunk.storetime.ge(chunk.charttime)].copy()
        # Recover admission time from 24h landmark; only first 24h accepted.
        chunk['hour']=(chunk.charttime-(chunk.landmark_time-pd.Timedelta(hours=24))).dt.total_seconds()/3600
        chunk=chunk.loc[chunk.hour.ge(0)&chunk.hour.lt(24)].copy()
        counters['available_by_landmark']+=len(chunk)
        if chunk.empty: continue
        chunk['lab']=chunk.itemid.map(LAB_MAP)
        chunk['valuenum']=pd.to_numeric(chunk.valuenum,errors='coerce')
        chunk=chunk.loc[np.isfinite(chunk.valuenum)].copy()
        # Retain units for audit; no silent unit conversion.
        for (lab,unit),n in chunk.groupby(['lab','valueuom'],dropna=False).size().items():
            key=str(lab)+' | '+str(unit)
            units[key]=units.get(key,0)+int(n)
        low=chunk.lab.map({k:v[0] for k,v in RANGES.items()})
        high=chunk.lab.map({k:v[1] for k,v in RANGES.items()})
        chunk=chunk.loc[chunk.valuenum.ge(low)&chunk.valuenum.le(high)].copy()
        counters['valid_numeric_unit_and_range']+=len(chunk)
        if not chunk.empty:
            selected.append(chunk[['subject_id','hadm_id','lab','charttime','storetime','hour','valuenum','valueuom']])
        if i%20==0: print(f'Chunks {i}; rows scanned {counters["rows_read"]:,}',flush=True)
    if not selected: raise RuntimeError('No eligible labs. Check paths, time constraints and source data.')
    labs=pd.concat(selected,ignore_index=True)
    del selected
    labs=labs.drop_duplicates(['subject_id','hadm_id','lab','charttime','storetime','valuenum','valueuom'])
    # Do not silently combine different measurement units for the same lab.
    # Save audit and fail if multiple non-null units occur for any lab.
    unit_report=pd.DataFrame([{'lab_unit':k,'count':v} for k,v in sorted(units.items())])
    unit_report.to_csv(output/'phase2_unit_audit.csv',index=False)
    mixed=labs.groupby('lab').valueuom.nunique(dropna=False)
    if (mixed>1).any():
        raise ValueError('Multiple units detected; review phase2_unit_audit.csv and harmonize before aggregating: '+str(mixed[mixed>1].to_dict()))
    labs=labs.sort_values(['hadm_id','lab','charttime','storetime'])
    # Four distinct temporal summaries per 6/6/12-hour bin, plus overall trend.
    labs['bin']=pd.cut(labs.hour,bins=[0,6,12,24],labels=['h00_06','h06_12','h12_24'],right=False,include_lowest=True)
    agg=labs.groupby(['hadm_id','lab','bin'],observed=True).valuenum.agg(['count','mean','min','max','std','last']).reset_index()
    agg=agg.rename(columns={'last':'latest'})
    wide=agg.pivot(index='hadm_id',columns=['lab','bin'],values=['count','mean','min','max','std','latest'])
    wide.columns=[f'{lab}_{timebin}_{stat}' for stat,lab,timebin in wide.columns]
    wide=wide.reset_index()
    overall=labs.groupby(['hadm_id','lab'],sort=False).agg(first=('valuenum','first'),latest=('valuenum','last'),
        first_hour=('hour','first'),last_hour=('hour','last'),n=('valuenum','size')).reset_index()
    overall['delta']=overall.latest-overall['first']
    elapsed=overall.last_hour-overall.first_hour
    overall['slope_per_hour']=np.where(elapsed.gt(0),overall.delta/elapsed,np.nan)
    trend=overall.pivot(index='hadm_id',columns='lab',values=['latest','delta','slope_per_hour','n'])
    trend.columns=[f'{lab}_overall_{stat}' for stat,lab in trend.columns]
    trend=trend.reset_index()
    features=pred.merge(wide,on='hadm_id',how='left',validate='one_to_one').merge(trend,on='hadm_id',how='left',validate='one_to_one')
    # Count features represent absence of measurement, not missing numeric results.
    count_cols=[c for c in features if c.endswith('_count') or c.endswith('_n')]
    features[count_cols]=features[count_cols].fillna(0)
    forbidden={'icu_transfer_24_72h','first_icu_intime','dischtime','deathtime','horizon_end','label_status'}
    if forbidden.intersection(features.columns): raise AssertionError('Outcome leakage columns detected')
    if len(features)!=len(pred) or features.hadm_id.duplicated().any(): raise AssertionError('Cohort cardinality changed')
    if labs.charttime.gt(labs.storetime).any(): raise AssertionError('Invalid chart/store ordering')
    if labs.hour.lt(0).any() or labs.hour.ge(24).any(): raise AssertionError('Out-of-window labs')
    features.to_parquet(output/'phase2_temporal_lab_features.parquet',index=False)
    report={**counters,'distinct_admissions_with_labs':int(labs.hadm_id.nunique()),
            'eligible_admissions':int(len(pred)),'feature_columns':int(len(features.columns)-len(pred.columns)),
            'retained_lab_rows_after_dedup':int(len(labs)),
            'policy':'charttime and storetime non-null and <= landmark; charttime within [0,24) hours; strict unit check'}
    (output/'phase2_report.json').write_text(json.dumps(report,indent=2))
    print(json.dumps(report,indent=2))
    print('Saved:',output/'phase2_temporal_lab_features.parquet')


In [ ]:
# Run phase2 (or reuse completed results).
if REUSE_EXISTING and ((OUTPUT_ROOT / 'phase2_outputs' / 'phase2_temporal_lab_features.parquet').exists()):
    print('Skipping phase2: expected outputs already exist')
else:
    run(DATA_ROOT, OUTPUT_ROOT / 'phase1_outputs', OUTPUT_ROOT / 'phase2_outputs', 500000)

**Check the output**

In [ ]:
p = OUTPUT_ROOT / 'phase2_outputs'
print('Output files:')
for f in sorted(p.iterdir()):
    print(' -', f.name)
for report in sorted(p.glob('*.json')):
    print('\n', report.name)
    print(report.read_text()[:4000])

## Phase 3 — Exploratory LightGBM and CatBoost ensemble

**Question:** Can nonlinear boosted trees rank eventual ICU transfers from the engineered features?

The code merges Phase 1 predictors, Phase 2 features and labels. It splits by **subject_id** to avoid sharing a patient's admissions across train/validation/test. The training partition determines which features have sufficient observed values; numeric missing values remain as NaN for the tree models. LightGBM and CatBoost use early stopping. The blend weight is chosen on validation AUPRC and evaluated once on test.

**Important:** This is the **original exploratory track**, kept for reproducibility. It duplicates three administrative predictors through the join (`*_x`, `*_y`); Phase 4B corrects this. Its test results were already examined, so do not use this track's test set for further architecture selection.

**Expected prior run:** test AUROC ~0.715 and AUPRC ~0.197, on its own test subset. This cannot be directly compared to Phase 4B because the partitions differ.

**Implementation:** The next code cell contains the complete original phase implementation (imports, constants, helper functions and model logic). It is intentionally visible and editable so you can inspect every operation.

In [ ]:
#!/usr/bin/env python3
"""Phase 3: leakage-aware grouped train/validation/test evaluation. Research only."""
import argparse, json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss, brier_score_loss
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from catboost import CatBoostClassifier

SEED=42

def metrics(y, p):
    return {'n':int(len(y)), 'positives':int(y.sum()), 'prevalence':float(y.mean()),
            'auroc':float(roc_auc_score(y,p)), 'auprc':float(average_precision_score(y,p)),
            'brier':float(brier_score_loss(y,p)), 'log_loss':float(log_loss(y,np.clip(p,1e-8,1-1e-8)))}

def main(args):
    out=Path(args.output); out.mkdir(parents=True,exist_ok=True)
    features=pd.read_parquet(Path(args.phase2)/'phase2_temporal_lab_features.parquet')
    base=pd.read_parquet(Path(args.phase1)/'phase1_landmark_predictors.parquet')
    labels=pd.read_parquet(Path(args.phase1)/'phase1_labels.parquet')
    for name,df in [('features',features),('baseline',base),('labels',labels)]:
        if df.hadm_id.isna().any() or df.hadm_id.duplicated().any(): raise ValueError(f'{name}: invalid or duplicated hadm_id')
    if 'subject_id' in features.columns:
        features=features.drop(columns='subject_id')
    if 'landmark_time' in features.columns:
        features=features.drop(columns='landmark_time')
    if 'subject_id' not in labels.columns: raise ValueError('Labels must contain subject_id')
    data=base.merge(features,on='hadm_id',how='left',validate='one_to_one')
    data=data.merge(labels[['subject_id','hadm_id','icu_transfer_24_72h']],on=['subject_id','hadm_id'],how='inner',validate='one_to_one')
    if len(data)!=len(base) or len(data)!=len(labels): raise ValueError('Row mismatch across phases')
    forbidden=['subject_id','hadm_id','landmark_time','icu_transfer_24_72h']
    X=data.drop(columns=forbidden)
    suspicious=[c for c in X if any(token in c.lower() for token in ['icu_transfer','icu_intime','dischtime','deathtime','horizon_end','label_status','hours_to_icu','first_icu'])]
    if suspicious: raise ValueError('Potential outcome leakage: '+str(suspicious))
    categorical=list(X.select_dtypes(include=['object','string','category']).columns)
    for col in categorical: X[col]=X[col].astype('string').fillna('__MISSING__').astype('category')
    numeric=[c for c in X if c not in categorical]
    X[numeric]=X[numeric].apply(pd.to_numeric,errors='raise').replace([np.inf,-np.inf],np.nan)
    all_null=[c for c in X if X[c].isna().all()]
    if all_null: X=X.drop(columns=all_null); categorical=[c for c in categorical if c not in all_null]
    y=data.icu_transfer_24_72h.astype(int)
    if set(y.unique())!={0,1}: raise ValueError('Expected binary outcome with both classes')
    groups=data.subject_id
    outer=GroupShuffleSplit(n_splits=1,test_size=.15,random_state=SEED)
    dev_idx,test_idx=next(outer.split(X,y,groups))
    inner=GroupShuffleSplit(n_splits=1,test_size=.15/.85,random_state=SEED+1)
    tr_sub,va_sub=next(inner.split(X.iloc[dev_idx],y.iloc[dev_idx],groups.iloc[dev_idx]))
    train_idx=dev_idx[tr_sub]; val_idx=dev_idx[va_sub]
    splits={'train':train_idx,'validation':val_idx,'test':test_idx}
    for a in splits:
        for b in splits:
            if a<b and set(groups.iloc[splits[a]]) & set(groups.iloc[splits[b]]): raise AssertionError('Patient leakage')
    for name,idx in splits.items():
        if y.iloc[idx].nunique()!=2: raise ValueError(f'{name} contains only one outcome class')
    Xtr,Xva,Xte=[X.iloc[splits[k]].copy() for k in ['train','validation','test']]
    ytr,yva,yte=[y.iloc[splits[k]] for k in ['train','validation','test']]
    # Missingness filter fit on training only. Native NaNs retained for both algorithms.
    keep=Xtr.columns[Xtr.notna().mean()>=.02].tolist()
    Xtr,Xva,Xte=[d[keep].copy() for d in (Xtr,Xva,Xte)]
    cats=[c for c in categorical if c in keep]
    lgb=LGBMClassifier(n_estimators=1200,learning_rate=.035,num_leaves=23,max_depth=-1,
        min_child_samples=70,colsample_bytree=.8,subsample=.8,subsample_freq=1,
        reg_lambda=5,random_state=SEED,n_jobs=args.threads,verbosity=-1)
    lgb.fit(Xtr,ytr,eval_set=[(Xva,yva)],eval_metric='average_precision',
            categorical_feature=cats,callbacks=[early_stopping(75,verbose=False),log_evaluation(0)])
    # CatBoost categorical inputs must be strings, not pandas category/NaN.
    def cb_frame(frame):
        z=frame.copy()
        for c in cats: z[c]=z[c].astype('string').fillna('__MISSING__').astype(str)
        return z
    cb=CatBoostClassifier(iterations=1400,depth=5,learning_rate=.04,l2_leaf_reg=6,
        loss_function='Logloss',eval_metric='PRAUC',random_seed=SEED,
        thread_count=args.threads,verbose=False)
    cb.fit(cb_frame(Xtr),ytr,cat_features=cats,eval_set=(cb_frame(Xva),yva),early_stopping_rounds=90,verbose=False)
    pv_l=lgb.predict_proba(Xva)[:,1]; pv_c=cb.predict_proba(cb_frame(Xva))[:,1]
    # Weight selected only on validation; no test-set model selection.
    weights=np.linspace(0,1,21)
    scored=[(float(average_precision_score(yva,w*pv_l+(1-w)*pv_c)),float(w)) for w in weights]
    best_ap,w=max(scored,key=lambda t:t[0])
    pt_l=lgb.predict_proba(Xte)[:,1]; pt_c=cb.predict_proba(cb_frame(Xte))[:,1]
    pred_val=w*pv_l+(1-w)*pv_c; pred_test=w*pt_l+(1-w)*pt_c
    report={'split':{k:{'admissions':len(idx),'patients':int(groups.iloc[idx].nunique()),'positives':int(y.iloc[idx].sum())} for k,idx in splits.items()},
      'predictors':len(keep),'categorical':cats,'dropped_high_missingness':len(X.columns)-len(keep),
      'best_lightgbm_iteration':int(lgb.best_iteration_), 'best_catboost_iteration':int(cb.best_iteration_),
      'blend_lightgbm_weight':w,'blend_catboost_weight':1-w,
      'validation':{'lightgbm':metrics(yva,pv_l),'catboost':metrics(yva,pv_c),'blend':metrics(yva,pred_val)},
      'test':{'lightgbm':metrics(yte,pt_l),'catboost':metrics(yte,pt_c),'blend':metrics(yte,pred_test)},
      'notes':['Retrospective renal ICD cohort may include diagnoses established after 24h.',
               'Observed ICU transfer is not equivalent to clinical ICU need.',
               'Test metrics should be inspected only once; subsequent model changes require a new holdout.']}
    (out/'phase3_report.json').write_text(json.dumps(report,indent=2))
    pd.DataFrame({'subject_id':data.iloc[test_idx].subject_id.values,'hadm_id':data.iloc[test_idx].hadm_id.values,
                  'outcome':yte.values,'lightgbm_probability':pt_l,'catboost_probability':pt_c,
                  'blend_probability':pred_test}).to_parquet(out/'phase3_test_predictions.parquet',index=False)
    pd.DataFrame({'feature':keep,'lightgbm_importance':lgb.feature_importances_}).sort_values('lightgbm_importance',ascending=False).to_csv(out/'phase3_feature_importance.csv',index=False)
    import joblib
    joblib.dump({'lightgbm':lgb,'catboost':cb,'columns':keep,'categorical':cats,'blend_lightgbm_weight':w},out/'phase3_models.joblib')
    print(json.dumps(report,indent=2))
    print('Saved outputs to',out)


In [ ]:
# Run phase3 (or reuse completed results).
if REUSE_EXISTING and ((OUTPUT_ROOT / 'phase3_outputs' / 'phase3_models.joblib').exists() and (OUTPUT_ROOT / 'phase3_outputs' / 'phase3_report.json').exists()):
    print('Skipping phase3: expected outputs already exist')
else:
    main(SimpleNamespace(phase1=str(OUTPUT_ROOT / 'phase1_outputs'), phase2=str(OUTPUT_ROOT / 'phase2_outputs'), output=str(OUTPUT_ROOT / 'phase3_outputs'), threads=THREADS))

**Check the output**

In [ ]:
p = OUTPUT_ROOT / 'phase3_outputs'
print('Output files:')
for f in sorted(p.iterdir()):
    print(' -', f.name)
for report in sorted(p.glob('*.json')):
    print('\n', report.name)
    print(report.read_text()[:4000])

## Phase 4 — Exploratory validation calibration diagnostics

**Question:** Do predicted probabilities agree with observed event frequencies?

This reconstructs Phase 3's patient-grouped validation subset and model predictions, verifies the original validation metrics, and plots observed versus predicted transfer rates in equal-count risk bins. The Brier score penalizes squared probability error; log loss penalizes confident incorrect probabilities. AUROC and AUPRC assess discrimination, not calibration.

**Crucial distinction:** The validation subset was already used for early stopping and ensemble weighting. These plots are **diagnostic only**, not unbiased estimates of the benefit of fitting a calibrator.

**Implementation:** The next code cell contains the complete original phase implementation (imports, constants, helper functions and model logic). It is intentionally visible and editable so you can inspect every operation.

In [ ]:
#!/usr/bin/env python3
"""Phase 4: exploratory validation calibration diagnostics (no test access).
Reconstructs Phase 3 grouped split exactly; never fits a calibrator.
"""
import argparse
import json
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             brier_score_loss, log_loss)

SEED = 42

def calibration_bins(y, p, n_bins=10):
    df = pd.DataFrame({'y': np.asarray(y), 'p': np.asarray(p)})
    df['bin'] = pd.qcut(df.p.rank(method='first'), q=n_bins, labels=False)
    return df.groupby('bin', as_index=False).agg(n=('y','size'), observed=('y','mean'), predicted=('p','mean'), positives=('y','sum'))

def main(args):
    out = Path(args.output); out.mkdir(parents=True, exist_ok=True)
    features = pd.read_parquet(Path(args.phase2) / 'phase2_temporal_lab_features.parquet')
    base = pd.read_parquet(Path(args.phase1) / 'phase1_landmark_predictors.parquet')
    labels = pd.read_parquet(Path(args.phase1) / 'phase1_labels.parquet')
    for name, df in [('features',features),('baseline',base),('labels',labels)]:
        if df.hadm_id.isna().any() or df.hadm_id.duplicated().any():
            raise ValueError(f'{name}: invalid or duplicated hadm_id')
    features = features.drop(columns=['subject_id','landmark_time'], errors='ignore')
    data = base.merge(features, on='hadm_id', how='left', validate='one_to_one')
    data = data.merge(labels[['subject_id','hadm_id','icu_transfer_24_72h']],
                      on=['subject_id','hadm_id'], how='inner', validate='one_to_one')
    if len(data) != len(base) or len(data) != len(labels):
        raise ValueError('Row mismatch')
    X = data.drop(columns=['subject_id','hadm_id','landmark_time','icu_transfer_24_72h'])
    suspicious = [c for c in X if any(t in c.lower() for t in
       ['icu_transfer','icu_intime','dischtime','deathtime','horizon_end','label_status','hours_to_icu','first_icu'])]
    if suspicious: raise ValueError('Potential leakage: '+str(suspicious))
    categorical = list(X.select_dtypes(include=['object','string','category']).columns)
    for col in categorical:
        X[col] = X[col].astype('string').fillna('__MISSING__').astype('category')
    numeric = [c for c in X if c not in categorical]
    X[numeric] = X[numeric].apply(pd.to_numeric, errors='raise').replace([np.inf,-np.inf], np.nan)
    X = X.drop(columns=[c for c in X if X[c].isna().all()])
    y = data.icu_transfer_24_72h.astype(int)
    groups = data.subject_id
    outer = GroupShuffleSplit(n_splits=1, test_size=.15, random_state=SEED)
    dev_idx, test_idx = next(outer.split(X,y,groups))
    inner = GroupShuffleSplit(n_splits=1, test_size=.15/.85, random_state=SEED+1)
    tr_sub, va_sub = next(inner.split(X.iloc[dev_idx], y.iloc[dev_idx], groups.iloc[dev_idx]))
    train_idx, val_idx = dev_idx[tr_sub], dev_idx[va_sub]
    if set(groups.iloc[train_idx]) & set(groups.iloc[val_idx]): raise AssertionError('Patient overlap')
    if set(groups.iloc[val_idx]) & set(groups.iloc[test_idx]): raise AssertionError('Patient overlap')
    if set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]): raise AssertionError('Patient overlap')
    bundle = joblib.load(Path(args.phase3) / 'phase3_models.joblib')
    cols = bundle['columns']; cats = bundle['categorical']
    missing = set(cols) - set(X.columns)
    if missing: raise ValueError('Missing saved model features: '+str(missing))
    Xv = X.iloc[val_idx][cols].copy()
    for c in cats:
        Xv[c] = Xv[c].astype('string').fillna('__MISSING__').astype('category')
    cb = Xv.copy()
    for c in cats: cb[c] = cb[c].astype('string').fillna('__MISSING__').astype(str)
    p_l = bundle['lightgbm'].predict_proba(Xv)[:,1]
    p_c = bundle['catboost'].predict_proba(cb)[:,1]
    w = float(bundle['blend_lightgbm_weight'])
    p = w*p_l+(1-w)*p_c
    yy = y.iloc[val_idx].to_numpy()
    old = json.loads((Path(args.phase3)/'phase3_report.json').read_text())
    if len(yy) != old['split']['validation']['admissions'] or int(yy.sum()) != old['split']['validation']['positives']:
        raise AssertionError('Reconstructed validation counts differ from Phase 3')
    ap = average_precision_score(yy,p)
    auc = roc_auc_score(yy,p)
    if not np.isclose(ap, old['validation']['blend']['auprc'], atol=1e-7) or not np.isclose(auc,old['validation']['blend']['auroc'],atol=1e-7):
        raise AssertionError('Predictions do not reproduce Phase 3 validation metrics; STOP')
    bins = calibration_bins(yy,p)
    bins.to_csv(out/'validation_reliability_bins.csv',index=False)
    pd.DataFrame({'subject_id':data.iloc[val_idx].subject_id.values,
                  'hadm_id':data.iloc[val_idx].hadm_id.values,
                  'outcome':yy,'blend_probability':p}).to_parquet(out/'validation_predictions.parquet',index=False)
    fig,ax=plt.subplots(figsize=(6,5))
    ax.plot([0,1],[0,1],'--',label='Perfect calibration')
    ax.plot(bins.predicted,bins.observed,'o-',label='Validation (10 equal-count bins)')
    ax.set(xlabel='Mean predicted ICU-transfer probability',ylabel='Observed ICU-transfer rate',title='Exploratory validation reliability')
    ax.legend(); fig.tight_layout(); fig.savefig(out/'validation_reliability.png',dpi=180);plt.close(fig)
    summary={'validation_admissions':int(len(yy)),'positives':int(yy.sum()),
             'prevalence':float(yy.mean()),'auroc':float(auc),'auprc':float(ap),
             'brier':float(brier_score_loss(yy,p)),
             'log_loss':float(log_loss(yy,np.clip(p,1e-8,1-1e-8))),
             'note':'Exploratory validation diagnostics only. Validation was used for early stopping and blend selection; no calibration model fitted. Test set not accessed.'}
    (out/'phase4_diagnostics.json').write_text(json.dumps(summary,indent=2))
    print(json.dumps(summary,indent=2))
    print('Saved to:',out)


In [ ]:
# Run phase4 (or reuse completed results).
if REUSE_EXISTING and ((OUTPUT_ROOT / 'phase4_outputs' / 'phase4_diagnostics.json').exists()):
    print('Skipping phase4: expected outputs already exist')
else:
    main(SimpleNamespace(phase1=str(OUTPUT_ROOT / 'phase1_outputs'), phase2=str(OUTPUT_ROOT / 'phase2_outputs'), phase3=str(OUTPUT_ROOT / 'phase3_outputs'), output=str(OUTPUT_ROOT / 'phase4_outputs')))

**Check the output**

In [ ]:
p = OUTPUT_ROOT / 'phase4_outputs'
print('Output files:')
for f in sorted(p.iterdir()):
    print(' -', f.name)
for report in sorted(p.glob('*.json')):
    print('\n', report.name)
    print(report.read_text()[:4000])

## Phase 4B — Cleaner training and independent calibration subsets

**Question:** Does calibration generalize when model fitting, early stopping, blending, calibration and evaluation use patient-disjoint subsets?

This track removes duplicate demographic/admission columns, builds five group-disjoint partitions, retrains LightGBM and CatBoost, chooses the ensemble weight separately from early stopping, and compares uncalibrated, sigmoid and isotonic probabilities. The calibrator is selected using the calibration subset, and evaluated on another subset.

**Interpretation:** Sigmoid calibration fits a logistic mapping; isotonic fits a flexible monotonic mapping. On the earlier run, isotonic was selected on calibration log loss, but the uncalibrated ensemble performed slightly better on evaluation Brier/log loss and had higher AUPRC. Preserve the preselected method in formal reporting; do not retroactively choose on evaluation outcomes.

**Expected prior run:** 553 features, 6,049 evaluation admissions, uncalibrated AUROC ~0.7154 and AUPRC ~0.1632. This is an internal reassessment, not external validation.

**Implementation:** The next code cell contains the complete original phase implementation (imports, constants, helper functions and model logic). It is intentionally visible and editable so you can inspect every operation.

In [ ]:
#!/usr/bin/env python3
"""Research-only renal ICU prediction: patient-disjoint development and calibration.
Usage: python3 phase4b_train_calibrate.py --phase1 PATH --phase2 PATH --output PATH
"""
import argparse
import json
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from catboost import CatBoostClassifier

SEED = 20261013

def partition(indices, groups, fraction, seed):
    splitter = GroupShuffleSplit(n_splits=1, test_size=fraction, random_state=seed)
    a, b = next(splitter.split(np.zeros(len(indices)), groups=groups.iloc[indices]))
    return indices[a], indices[b]

def evaluate(y, p):
    p = np.clip(np.asarray(p, dtype=float), 1e-8, 1 - 1e-8)
    return dict(n=int(len(y)), positives=int(np.sum(y)), prevalence=float(np.mean(y)),
                auroc=float(roc_auc_score(y,p)), auprc=float(average_precision_score(y,p)),
                brier=float(brier_score_loss(y,p)), log_loss=float(log_loss(y,p)))

def cb_frame(frame, cats):
    result = frame.copy()
    for col in cats:
        result[col] = result[col].astype('string').fillna('__MISSING__').astype(str)
    return result

def main(args):
    output = Path(args.output)
    output.mkdir(parents=True, exist_ok=True)
    base = pd.read_parquet(Path(args.phase1) / 'phase1_landmark_predictors.parquet')
    labs = pd.read_parquet(Path(args.phase2) / 'phase2_temporal_lab_features.parquet')
    labels = pd.read_parquet(Path(args.phase1) / 'phase1_labels.parquet')
    for name, table in [('base',base), ('labs',labs), ('labels',labels)]:
        if table.hadm_id.isna().any() or table.hadm_id.duplicated().any():
            raise ValueError(f'{name}: missing or duplicate hadm_id')
    labs = labs.drop(columns=['subject_id','landmark_time'], errors='ignore')
    # Base demographics are canonical: eliminate redundant columns from Phase 2.
    overlap = set(labs.columns).intersection(base.columns) - {'hadm_id'}
    labs = labs.drop(columns=sorted(overlap))
    data = base.merge(labs, on='hadm_id', how='left', validate='one_to_one')
    data = data.merge(labels[['subject_id','hadm_id','icu_transfer_24_72h']],
                      on=['subject_id','hadm_id'], how='inner', validate='one_to_one')
    if len(data) != len(base) or len(data) != len(labels):
        raise ValueError('Row count mismatch')
    target = 'icu_transfer_24_72h'
    forbidden = ['subject_id','hadm_id','landmark_time',target]
    X = data.drop(columns=forbidden)
    suspicious = [c for c in X if any(t in c.lower() for t in
                  ['icu_transfer','icu_intime','dischtime','deathtime','horizon_end',
                   'label_status','hours_to_icu','first_icu'])]
    if suspicious:
        raise ValueError('Possible outcome leakage: ' + str(suspicious))
    y = data[target].astype(int)
    groups = data.subject_id
    cats = list(X.select_dtypes(include=['object','string','category']).columns)
    for c in cats:
        X[c] = X[c].astype('string').fillna('__MISSING__').astype('category')
    numeric = [c for c in X if c not in cats]
    X[numeric] = X[numeric].apply(pd.to_numeric, errors='raise').replace([np.inf,-np.inf],np.nan)
    all_indices = np.arange(len(data))
    # Patient-disjoint target fractions: train 55%, early-stop 15%, blend 10%, calibration 10%, evaluation 10%.
    remaining, test = partition(all_indices, groups, .10, SEED)
    remaining, calibration = partition(remaining, groups, .10/.90, SEED+1)
    remaining, blend = partition(remaining, groups, .10/.80, SEED+2)
    train, early = partition(remaining, groups, .15/.70, SEED+3)
    splits = {'train':train,'early_stop':early,'blend':blend,'calibration':calibration,'evaluation':test}
    for a, ia in splits.items():
        if y.iloc[ia].nunique() != 2:
            raise ValueError(f'{a}: only one outcome class')
        for b, ib in splits.items():
            if a < b and set(groups.iloc[ia]) & set(groups.iloc[ib]):
                raise AssertionError(f'Patient overlap: {a}, {b}')
    keep = X.iloc[train].columns[X.iloc[train].notna().mean() >= .02].tolist()
    cats = [c for c in cats if c in keep]
    X = X[keep]
    xtr, xearly, xblend, xcal, xte = [X.iloc[splits[k]].copy() for k in splits]
    ytr, yearly, yblend, ycal, yte = [y.iloc[splits[k]] for k in splits]
    lgb = LGBMClassifier(n_estimators=1200,learning_rate=.035,num_leaves=23,
        min_child_samples=70,colsample_bytree=.8,subsample=.8,subsample_freq=1,
        reg_lambda=5,random_state=SEED,n_jobs=args.threads,verbosity=-1)
    lgb.fit(xtr,ytr,eval_set=[(xearly,yearly)],eval_metric='average_precision',
            categorical_feature=cats,callbacks=[early_stopping(75,verbose=False),log_evaluation(0)])
    cat = CatBoostClassifier(iterations=1400,depth=5,learning_rate=.04,l2_leaf_reg=6,
        loss_function='Logloss',eval_metric='PRAUC',random_seed=SEED,
        thread_count=args.threads,verbose=False)
    cat.fit(cb_frame(xtr,cats),ytr,cat_features=cats,
            eval_set=(cb_frame(xearly,cats),yearly),early_stopping_rounds=90,verbose=False)
    def both(frame):
        return (lgb.predict_proba(frame)[:,1],cat.predict_proba(cb_frame(frame,cats))[:,1])
    pb_l,pb_c = both(xblend)
    scores = [(average_precision_score(yblend,w*pb_l+(1-w)*pb_c),float(w))
              for w in np.linspace(0,1,21)]
    _, weight = max(scores,key=lambda item:item[0])
    pc_l,pc_c = both(xcal)
    pc = weight*pc_l + (1-weight)*pc_c
    # Platt scaling on log-odds; independent calibration set.
    logits = np.log(np.clip(pc,1e-6,1-1e-6) / (1-np.clip(pc,1e-6,1-1e-6)))
    sigmoid = LogisticRegression(C=1e3,max_iter=1000,random_state=SEED)
    sigmoid.fit(logits.reshape(-1,1),ycal)
    isotonic = IsotonicRegression(y_min=0,y_max=1,out_of_bounds='clip')
    isotonic.fit(pc,ycal)
    sigmoid_cal = sigmoid.predict_proba(logits.reshape(-1,1))[:,1]
    isotonic_cal = isotonic.predict(pc)
    # Select calibrator ONLY using calibration data. This comparison is optimistic;
    # independent evaluation split is used once below for reporting.
    candidates = {'uncalibrated':pc,'sigmoid':sigmoid_cal,'isotonic':isotonic_cal}
    selected = min(candidates,key=lambda k:log_loss(ycal,np.clip(candidates[k],1e-8,1-1e-8)))
    pt_l,pt_c = both(xte)
    pt = weight*pt_l+(1-weight)*pt_c
    pt_logit = np.log(np.clip(pt,1e-6,1-1e-6)/(1-np.clip(pt,1e-6,1-1e-6)))
    test_probs = {'uncalibrated':pt,'sigmoid':sigmoid.predict_proba(pt_logit.reshape(-1,1))[:,1],
                  'isotonic':isotonic.predict(pt)}
    report = {'split':{k:{'admissions':int(len(v)),'patients':int(groups.iloc[v].nunique()),
                           'positives':int(y.iloc[v].sum())} for k,v in splits.items()},
              'features':len(keep),'categorical':cats,
              'best_iterations':{'lightgbm':int(lgb.best_iteration_),'catboost':int(cat.best_iteration_)},
              'lightgbm_blend_weight':weight,'selected_calibration':selected,
              'calibration_metrics':{k:evaluate(ycal,v) for k,v in candidates.items()},
              'evaluation_metrics':{k:evaluate(yte,v) for k,v in test_probs.items()},
              'limitations':['Retrospective diagnosis-defined cohort',
                 'Observed ICU transfer does not equal clinical need',
                 'Previously inspected MIMIC-IV test outcomes; this is an internal reassessment, not an untouched external validation',
                 'Calibrator selection on calibration data may be optimistic; evaluation set was not used for selection']}
    (output/'phase4b_report.json').write_text(json.dumps(report,indent=2))
    joblib.dump({'lightgbm':lgb,'catboost':cat,'columns':keep,'categorical':cats,
                 'blend_lightgbm_weight':weight,'sigmoid':sigmoid,'isotonic':isotonic,
                 'selected_calibration':selected},output/'phase4b_models.joblib')
    pd.DataFrame({'subject_id':data.iloc[test].subject_id.values,
                  'hadm_id':data.iloc[test].hadm_id.values,'outcome':yte.values,
                  **{k+'_probability':v for k,v in test_probs.items()}}).to_parquet(
                      output/'phase4b_evaluation_predictions.parquet',index=False)
    fig,ax=plt.subplots(figsize=(7,6))
    ax.plot([0,1],[0,1],'--',label='Perfect calibration')
    for name,p in test_probs.items():
        d=pd.DataFrame({'p':p,'y':yte.to_numpy()})
        d['bin']=pd.qcut(d['p'].rank(method='first'),10,labels=False)
        bins=d.groupby('bin').agg(predicted=('p','mean'),observed=('y','mean'))
        ax.plot(bins.predicted,bins.observed,'o-',label=name)
    ax.set(xlabel='Mean predicted probability',ylabel='Observed ICU-transfer rate',
           title='Internal evaluation reliability (10 equal-count bins)')
    ax.legend();fig.tight_layout();fig.savefig(output/'phase4b_reliability.png',dpi=200)
    plt.close(fig)
    print(json.dumps(report,indent=2))
    print('Saved to:',output)


In [ ]:
# Run phase4b (or reuse completed results).
if REUSE_EXISTING and ((OUTPUT_ROOT / 'phase4b_outputs' / 'phase4b_models.joblib').exists() and (OUTPUT_ROOT / 'phase4b_outputs' / 'phase4b_report.json').exists()):
    print('Skipping phase4b: expected outputs already exist')
else:
    main(SimpleNamespace(phase1=str(OUTPUT_ROOT / 'phase1_outputs'), phase2=str(OUTPUT_ROOT / 'phase2_outputs'), output=str(OUTPUT_ROOT / 'phase4b_outputs'), threads=THREADS))

**Check the output**

In [ ]:
p = OUTPUT_ROOT / 'phase4b_outputs'
print('Output files:')
for f in sorted(p.iterdir()):
    print(' -', f.name)
for report in sorted(p.glob('*.json')):
    print('\n', report.name)
    print(report.read_text()[:4000])

## Phase 5 — Explainability with SHAP

**Question:** Which features influence the two trained models' risk estimates?

SHAP assigns model-specific feature contributions relative to a baseline prediction. This phase reconstructs the model input schema, samples evaluation admissions, and generates global importance and patient-level explanations for LightGBM and CatBoost. Compare the rankings and inspect reliance on admission pathways, renal laboratory values and monitoring intensity.

**Interpretation limitations:** The SHAP contributions are in **model margin units**, not direct changes in blended probability. Do not average the two models' margin SHAP values and label the result an exact explanation of the ensemble probability. SHAP is associational, not causal.

Earlier plots showed admission type/location, WBC and creatinine as prominent features. An administrative-feature ablation is a valuable next comparison.

**Implementation:** The next code cell contains the complete original phase implementation (imports, constants, helper functions and model logic). It is intentionally visible and editable so you can inspect every operation.

In [ ]:
#!/usr/bin/env python3
"""Phase 5: retrospective SHAP explanations for the raw LightGBM/CatBoost ensemble."""
import argparse, json
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit

SEED=42

def main(a):
    import shap
    out=Path(a.output); out.mkdir(parents=True,exist_ok=True)
    bundle=joblib.load(Path(a.phase4b)/'phase4b_models.joblib')
    cols=bundle['columns']; cats=bundle['categorical']; w=float(bundle['blend_lightgbm_weight'])
    base=pd.read_parquet(Path(a.phase1)/'phase1_landmark_predictors.parquet')
    features=pd.read_parquet(Path(a.phase2)/'phase2_temporal_lab_features.parquet')
    labels=pd.read_parquet(Path(a.phase1)/'phase1_labels.parquet')
    for name,df in [('base',base),('features',features),('labels',labels)]:
        if df.hadm_id.isna().any() or df.hadm_id.duplicated().any(): raise ValueError(f'{name} invalid hadm_id')
    features=features.drop(columns=['subject_id','landmark_time'],errors='ignore')
    # Match the Phase 4B training feature schema; duplicate baseline fields from Phase 2 are ignored.
    duplicates=[c for c in features if c in base.columns and c!='hadm_id']
    features=features.drop(columns=duplicates)
    data=base.merge(features,on='hadm_id',how='left',validate='one_to_one').merge(
        labels[['subject_id','hadm_id','icu_transfer_24_72h']],on=['subject_id','hadm_id'],how='inner',validate='one_to_one')
    if len(data)!=len(base): raise ValueError('Cohort size changed')
    X=data.reindex(columns=cols).copy()
    if X.columns.tolist()!=cols: raise AssertionError('Feature order mismatch')
    if X.columns.duplicated().any(): raise ValueError('Duplicate features')
    for c in cats: X[c]=X[c].astype('string').fillna('__MISSING__').astype('category')
    for c in X.columns.difference(cats): X[c]=pd.to_numeric(X[c],errors='raise').replace([np.inf,-np.inf],np.nan)
    # Verify model-compatible feature types and avoid missing columns silently.
    missing=set(cols)-set(data.columns)
    if missing: raise ValueError('Missing model features: '+str(sorted(missing)[:20]))
    # Sample only patients in the saved evaluation predictions (not used to tune models).
    pred=pd.read_parquet(Path(a.phase4b)/'phase4b_evaluation_predictions.parquet')
    if 'hadm_id' not in pred: raise ValueError('Evaluation predictions missing hadm_id')
    indices=data.index[data.hadm_id.isin(pred.hadm_id)]
    if len(indices)!=len(pred): raise ValueError('Evaluation IDs do not match reconstructed cohort')
    rng=np.random.default_rng(SEED)
    indices=rng.choice(indices,size=min(a.sample,len(indices)),replace=False)
    sample=X.loc[indices].copy(); info=data.loc[indices,['subject_id','hadm_id','icu_transfer_24_72h']].copy()
    cb=sample.copy()
    for c in cats: cb[c]=cb[c].astype('string').fillna('__MISSING__').astype(str)
    lgb=bundle['lightgbm']; cat=bundle['catboost']
    p_l=lgb.predict_proba(sample)[:,1]; p_c=cat.predict_proba(cb)[:,1]
    raw=w*p_l+(1-w)*p_c
    # SHAP values in raw margin space: separate model explanations, NOT additively composable
    # in probability space. Use model-specific SHAP, and ensemble probability separately.
    lgb_ex=shap.TreeExplainer(lgb)
    lv=lgb_ex.shap_values(sample)
    if isinstance(lv,list): lv=lv[-1]
    lv=np.asarray(lv)
    if lv.ndim==3: lv=lv[:,:,1]
    if lv.shape!=(len(sample),len(cols)): raise ValueError(f'Unexpected SHAP shape {lv.shape}')
    # Categorical string features can cause problems with CatBoost SHAP APIs; native CatBoost
    # SHAP uses its Pool and supports categorical values.
    from catboost import Pool
    pool=Pool(cb,cat_features=cats)
    cv=np.asarray(cat.get_feature_importance(pool,type='ShapValues'))
    if cv.shape!=(len(sample),len(cols)+1): raise ValueError(f'Unexpected CatBoost SHAP shape {cv.shape}')
    cat_vals=cv[:,:-1]
    def rank(values,name):
        df=pd.DataFrame({'feature':cols,'mean_abs_shap_margin':np.abs(values).mean(axis=0)}).sort_values('mean_abs_shap_margin',ascending=False)
        df.to_csv(out/f'{name}_shap_importance.csv',index=False)
        top=df.head(20).iloc[::-1]
        fig,ax=plt.subplots(figsize=(10,8));ax.barh(top.feature,top.mean_abs_shap_margin)
        ax.set_xlabel('Mean absolute SHAP value (model margin units)');ax.set_title(name+' feature importance')
        fig.tight_layout();fig.savefig(out/f'{name}_importance.png',dpi=180);plt.close(fig)
        return df
    rank(lv,'lightgbm');rank(cat_vals,'catboost')
    # Numerical feature beeswarm (categorical codes deliberately not fabricated).
    numeric=[c for c in cols if c not in cats]
    idx=[cols.index(c) for c in numeric]
    shap.summary_plot(lv[:,idx],sample[numeric],show=False,max_display=20)
    plt.title('LightGBM SHAP summary (margin units)');plt.tight_layout()
    plt.savefig(out/'lightgbm_beeswarm.png',dpi=180,bbox_inches='tight');plt.close('all')
    # Renal-feature dependence plots for features actually in the trained model.
    for term in ['creatinine','bun','potassium','anion_gap']:
        choices=[c for c in numeric if term in c and ('latest' in c or 'mean' in c)]
        if not choices: continue
        feature=choices[0]; j=cols.index(feature)
        fig,ax=plt.subplots(figsize=(7,5));ax.scatter(sample[feature],lv[:,j],s=10,alpha=.4)
        ax.set_xlabel(feature);ax.set_ylabel('LightGBM SHAP (margin)');ax.set_title(feature+' dependence')
        fig.tight_layout();fig.savefig(out/f'dependence_{term}.png',dpi=180);plt.close(fig)
    result=info.reset_index(drop=True).copy()
    result['lightgbm_probability']=p_l;result['catboost_probability']=p_c;result['raw_ensemble_probability']=raw
    result.to_parquet(out/'phase5_sample_predictions.parquet',index=False)
    # Patient-specific contributions for each model, rather than mathematically invalid
    # weighted averaging of log-odds SHAP values.
    for tag,pos in [('highest',int(np.argmax(raw))),('lowest',int(np.argmin(raw)))]:
        for model,vals in [('lightgbm',lv),('catboost',cat_vals)]:
            ix=np.argsort(np.abs(vals[pos]))[-15:][::-1]
            pd.DataFrame({'feature':np.array(cols)[ix],'shap_margin':vals[pos,ix]}).to_csv(out/f'{tag}_{model}_explanation.csv',index=False)
    report={'sample_n':len(sample),'model_features':len(cols),'lightgbm_weight':w,
        'raw_ensemble_mean_probability':float(raw.mean()),'observed_event_rate':float(info.icu_transfer_24_72h.mean()),
        'interpretation':'Model-specific SHAP values are in margin units and must not be averaged to explain blended probability. SHAP is associational, not causal.',
        'note':'Research-only retrospective explanations. No protected patient information included in plots.'}
    (out/'phase5_report.json').write_text(json.dumps(report,indent=2))
    print(json.dumps(report,indent=2));print('Saved to',out)


In [ ]:
# Run phase5 (or reuse completed results).
if REUSE_EXISTING and ((OUTPUT_ROOT / 'phase5_outputs' / 'phase5_report.json').exists()):
    print('Skipping phase5: expected outputs already exist')
else:
    main(SimpleNamespace(phase1=str(OUTPUT_ROOT / 'phase1_outputs'), phase2=str(OUTPUT_ROOT / 'phase2_outputs'), phase4b=str(OUTPUT_ROOT / 'phase4b_outputs'), output=str(OUTPUT_ROOT / 'phase5_outputs'), sample=SHAP_SAMPLE))

**Check the output**

In [ ]:
p = OUTPUT_ROOT / 'phase5_outputs'
print('Output files:')
for f in sorted(p.iterdir()):
    print(' -', f.name)
for report in sorted(p.glob('*.json')):
    print('\n', report.name)
    print(report.read_text()[:4000])

## Phase 6 — Retrospective capacity-constrained ranking simulation

**Question:** Under a hypothetical selection budget, how many eventual ICU transfers does risk ranking capture compared with random selection?

The code ranks evaluation admissions by the **uncalibrated** ensemble probability and evaluates capacity budgets of 5%, 10%, 15%, 20% and 30% of the evaluation sample. It reports precision at capacity, recall at capacity, and capture lift over random allocation; random selection is also simulated for uncertainty intervals.

**Very important:** These admissions occurred at different times. They were **not competing simultaneously for real ICU beds**. With one identical-cost resource constraint, maximizing the sum of predicted risks reduces to top-k sorting. This is not a true hospital operations optimizer, a clinical triage policy or an estimate of lives saved.

**Implementation:** The next code cell contains the complete original phase implementation (imports, constants, helper functions and model logic). It is intentionally visible and editable so you can inspect every operation.

In [ ]:
#!/usr/bin/env python3
"""Phase 6: retrospective ICU-transfer capture under hypothetical capacity.
This is NOT a clinical allocation or treatment-benefit model.
"""
import argparse
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

CAPACITIES = [0.05, 0.10, 0.15, 0.20, 0.30]

def choose_col(df, requested, candidates, purpose):
    if requested:
        if requested not in df.columns:
            raise ValueError(f'{purpose} column {requested!r} not found. Available: {list(df.columns)}')
        return requested
    matches = [c for c in candidates if c in df.columns]
    if len(matches) != 1:
        raise ValueError(f'Cannot unambiguously identify {purpose} column. Candidates present: {matches}. Available: {list(df.columns)}. Pass explicit --{purpose}-col.')
    return matches[0]

def main():
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument('--phase4b', required=True, help='Directory containing phase4b_evaluation_predictions.parquet')
    ap.add_argument('--output', required=True)
    ap.add_argument('--risk-col', default=None)
    ap.add_argument('--outcome-col', default=None)
    ap.add_argument('--random-repeats', type=int, default=1000)
    ap.add_argument('--seed', type=int, default=42)
    args = ap.parse_args()
    if args.random_repeats < 1:
        raise ValueError('--random-repeats must be >=1')
    out = Path(args.output)
    out.mkdir(parents=True, exist_ok=True)
    infile = Path(args.phase4b) / 'phase4b_evaluation_predictions.parquet'
    df = pd.read_parquet(infile)
    print('Available columns:', df.columns.tolist())
    risk_col = choose_col(df, args.risk_col, ['uncalibrated_probability', 'raw_ensemble_probability', 'blend_probability', 'uncalibrated', 'raw_probability', 'ensemble_probability', 'probability_uncalibrated'], 'risk')
    outcome_col = choose_col(df, args.outcome_col, ['outcome', 'icu_transfer_24_72h', 'icu_transfer', 'label', 'y_true'], 'outcome')
    if df.empty or df[[risk_col, outcome_col]].isna().any().any():
        raise ValueError('Empty dataset or missing outcome/risk values')
    if 'hadm_id' in df and df.hadm_id.duplicated().any():
        raise ValueError('Duplicate hadm_id: expected one row per admission')
    p = pd.to_numeric(df[risk_col], errors='raise').to_numpy(dtype=float)
    y = pd.to_numeric(df[outcome_col], errors='raise').to_numpy(dtype=float)
    if not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
        raise ValueError('Predicted risks must be finite probabilities in [0,1]')
    if not np.isin(y, [0, 1]).all():
        raise ValueError('Outcomes must be binary 0/1')
    y = y.astype(int)
    n, positives = len(y), int(y.sum())
    if positives == 0:
        raise ValueError('No positive outcomes in evaluation dataset')
    rng = np.random.default_rng(args.seed)
    # Stable ranking; ties resolved by row order (not by observed outcome).
    order = np.argsort(-p, kind='stable')
    rows = []
    selected = []
    for fraction in CAPACITIES:
        k = max(1, int(np.floor(n * fraction)))
        chosen = order[:k]
        hits = int(y[chosen].sum())
        random_hits = np.empty(args.random_repeats, dtype=int)
        for j in range(args.random_repeats):
            sample = rng.choice(n, size=k, replace=False)
            random_hits[j] = int(y[sample].sum())
        rows.append({
            'capacity_fraction': fraction, 'capacity_admissions': k,
            'total_admissions': n, 'total_eventual_transfers': positives,
            'risk_ranked_captured': hits,
            'risk_ranked_precision_at_capacity': hits/k,
            'risk_ranked_recall_at_capacity': hits/positives,
            'random_expected_captured_analytic': k*positives/n,
            'random_mean_captured_simulated': float(random_hits.mean()),
            'random_captured_p025': float(np.quantile(random_hits, 0.025)),
            'random_captured_p975': float(np.quantile(random_hits, 0.975)),
            'random_mean_precision_at_capacity': float(random_hits.mean()/k),
            'random_mean_recall_at_capacity': float(random_hits.mean()/positives),
            'capture_lift_vs_random_expectation': float(hits/(k*positives/n)),
        })
        for rank, idx in enumerate(chosen, start=1):
            selected.append({'capacity_fraction': fraction, 'rank': rank,
                             'hadm_id': df.iloc[idx]['hadm_id'] if 'hadm_id' in df else int(idx),
                             'predicted_risk': float(p[idx]), 'observed_transfer': int(y[idx])})
    result = pd.DataFrame(rows)
    result.to_csv(out/'phase6_capacity_comparison.csv', index=False)
    pd.DataFrame(selected).to_csv(out/'phase6_ranked_selection_audit.csv', index=False)
    fig, ax = plt.subplots(figsize=(8,5))
    x = result['capacity_fraction'].to_numpy()*100
    ax.plot(x, result['risk_ranked_recall_at_capacity'], marker='o', label='Risk ranking')
    ax.plot(x, result['random_mean_recall_at_capacity'], marker='s', label='Random allocation (mean)')
    ax.fill_between(x, result['random_captured_p025']/positives, result['random_captured_p975']/positives, alpha=.15, label='Random 95% simulation interval')
    ax.set(xlabel='Hypothetical capacity (% of evaluation admissions)', ylabel='Fraction of eventual ICU transfers captured', title='Retrospective event capture by hypothetical capacity')
    ax.set_ylim(0,1)
    ax.legend()
    fig.tight_layout()
    fig.savefig(out/'phase6_capture_curve.png', dpi=220)
    plt.close(fig)
    report = {'source_file': str(infile), 'risk_column': risk_col, 'outcome_column': outcome_col,
              'evaluation_admissions': n, 'eventual_transfers': positives,
              'prevalence': positives/n, 'random_repeats': args.random_repeats,
              'note': 'This is a retrospective ranking simulation. It does not model real ICU bed occupancy, clinical eligibility, resource needs, competing risks, patient outcomes under alternative treatment, or lives saved. A one-resource equal-cost linear allocation objective reduces to top-k risk ranking.'}
    (out/'phase6_report.json').write_text(json.dumps(report, indent=2))
    print('\n'+result[['capacity_fraction','capacity_admissions','risk_ranked_captured','risk_ranked_precision_at_capacity','risk_ranked_recall_at_capacity','random_expected_captured_analytic','capture_lift_vs_random_expectation']].to_string(index=False))
    print('\nSaved outputs to', out)


In [ ]:
# Run phase6 (or reuse completed results).
if REUSE_EXISTING and ((OUTPUT_ROOT / 'phase6_outputs' / 'phase6_report.json').exists()):
    print('Skipping phase6: expected outputs already exist')
else:
    import sys
    _original_argv = sys.argv[:]
    try:
        sys.argv = ['phase6_allocation.py', '--phase4b', str(OUTPUT_ROOT / 'phase4b_outputs'), '--output', str(OUTPUT_ROOT / 'phase6_outputs')]
        main()
    finally:
        sys.argv = _original_argv

**Check the output**

In [ ]:
p = OUTPUT_ROOT / 'phase6_outputs'
print('Output files:')
for f in sorted(p.iterdir()):
    print(' -', f.name)
for report in sorted(p.glob('*.json')):
    print('\n', report.name)
    print(report.read_text()[:4000])

## Comparing additional ML architectures fairly

For architecture comparisons, reuse the **same Phase 1 labels, Phase 2 feature table, and patient-grouped splits**. Train candidate models using only training data; use early-stopping/validation data for hyperparameters; choose any ensemble/calibration mapping without looking at final evaluation outcomes. Keep identical prediction horizons and report AUROC, AUPRC, Brier, log loss, calibration curves and precision/recall at a fixed alert budget. Include a simple baseline.

**Suggested candidates:** logistic regression (after train-only imputation/encoding/scaling), random forest, XGBoost, LightGBM, CatBoost, and a temporal neural network only if its sequential input representation is built without post-landmark information. A model comparison table should also include prevalence, sample size, confidence intervals and computational cost.

**Methodological work still needed:** diagnosis availability at hour 24; handling of early discharge/death and censoring; subgroup robustness; ablation of admission type/location; assessment of measurement frequency as a workflow proxy; and truly time-aware operational simulation. The Phase 6 top-k policy is not an ICU resource optimizer.